# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Avinish4945/flyRank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

One row represents one search query for one landing page on one day.

### Time Window

This notebook uses data from March 2026 (month = '2026-03') because it is a mid-panel month and avoids the final month (June 2026), which is reserved as the test period.

In [19]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [21]:
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE strftime(report_date, '%Y-%m') = '2026-03';
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

- gsc_clicks
- gsc_impressions
- gsc_avg_position
- ga4_sessions
- ga4_engagement_rate

These features are available before making a prediction and describe historical search performance.

## Label

- Future content performance (or another prediction target derived later).

These values are not used as input features.

## Context

- client_id
- content_id
- report_date

These fields identify the row and are used for grouping, filtering, or joining, not for training.

## Excluded

- Future information from June 2026.
- Label-derived fields.
- Rows where `ga4_data_available = FALSE`.

Reason: Including future information or unavailable GA4 values would introduce data leakage or misleading signals.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [23]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE strftime(report_date,'%Y-%m')='2026-03'
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows
0,9841378,413966


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification

The following queries verify that:

1. One row represents one client, one content item, and one report date.
2. The selected data covers March 2026.
3. Only rows with available GA4 data are used where appropriate.

In [24]:
grain_query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""

con.sql(grain_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


In [25]:
count_query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
"""

con.sql(count_query).df()


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [26]:
availability_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
"""

con.sql(availability_query).df()

,total_rows,ga4_available_rows
0,9841378,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset has the following limitations:

1. The analysis is limited to historical Search Console (GSC) and Google Analytics 4 (GA4) data and does not include external factors such as algorithm updates, competitor activity, or seasonal trends.

2. Some clients do not have complete GA4 history. Rows where `ga4_data_available = FALSE` contain zero-filled GA4 values, so these rows should be filtered instead of treating zeros as real engagement.

3. This notebook uses only the March 2026 partition for analysis. Results from a single month may not generalize to all time periods.

4. The final month (June 2026) is intentionally excluded because it is reserved as the outcome/test period. Using it for feature development could introduce data leakage.

### Output

This notebook produces a verified data contract describing the unit of analysis, time window, field classifications, verification queries, and known limitations before any feature engineering or model training.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
limits_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
"""

con.sql(limits_query).df()

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.